In [ ]:
%%capture

!pip install psycopg2-binary

In [ ]:
import psycopg2

In [ ]:
import os
# Set the database credentials
host = os.environ['DB_HOST']
port = '5432'
user = os.environ['DB_USER']
password = os.environ['DB_PASSWORD']
database = 'manas_users'

In [ ]:
connection = psycopg2.connect(
    host=host,
    port=port,
    user=user,
    password=password,
    database=database
)
print('Connected to the database')

In [ ]:
# prompt: print version after connecting to rds database

cursor = connection.cursor()
cursor.execute("SELECT version();")
record = cursor.fetchone()
print("You are connected to - ", record, "\n")

In [ ]:
# prompt: select manas_users database which was created while creating db instance
try:
  cursor.execute("SELECT * FROM manas_users;")
  records = cursor.fetchall()

  # Print the results
  for row in records:
      print(row)

except psycopg2.Error as e:
    print(f"Error executing query: {e}")
    connection.rollback()

In [ ]:
import os
from sqlalchemy import create_engine, Column, Integer, String, Boolean, Date, ForeignKey
from sqlalchemy.orm import sessionmaker, declarative_base

# Replace with your RDS credentials
DB_USER = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]
DB_HOST = os.environ["DB_HOST"]
DB_PORT = "5432"
DB_NAME = "manas_users"

# Create DB Engine
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

Base = declarative_base()

# Define Tables
class Roles(Base):
    __tablename__ = "roles"
    role_id = Column(Integer, primary_key=True, autoincrement=True)
    role_name = Column(String(255), nullable=False)

class University(Base):
    __tablename__ = "university"
    university_id = Column(Integer, primary_key=True, autoincrement=True)
    name = Column(String(255), nullable=False)

class ManasUser(Base):
    __tablename__ = "manas_user"
    man_user_id = Column(Integer, primary_key=True, autoincrement=True)
    role_id = Column(Integer, ForeignKey("roles.role_id"))
    first_name = Column(String(255), nullable=False)
    last_name = Column(String(255), nullable=False)
    email = Column(String(255), unique=True, nullable=False)
    profile_photo = Column(String)
    university_id = Column(Integer, ForeignKey("university.university_id"))
    no_students = Column(Integer)
    practice_type = Column(String(255))
    license_no = Column(String(255))
    organization = Column(String(255))

class AlfredaUser(Base):
    __tablename__ = "alfreda_user"
    fred_user_id = Column(Integer, primary_key=True, autoincrement=True)
    role_id = Column(Integer, ForeignKey("roles.role_id"))
    first_name = Column(String(255), nullable=False)
    last_name = Column(String(255), nullable=False)
    email = Column(String(255), unique=True, nullable=False)
    preferred_name = Column(String(255))
    pronoun = Column(String(50))
    birth_date = Column(Date)
    consent = Column(Boolean)
    progress_sum = Column(Integer)
    checkin_freq = Column(Integer)
    mental_health = Column(String)
    university_id = Column(Integer, ForeignKey("university.university_id"))
    major = Column(String(255))
    academic_year = Column(String(50))
    sports = Column(String(255))
    sports_level = Column(String(50))
    training_hours = Column(Integer)
    active_mentor = Column(Boolean)

# Create Tables
Base.metadata.create_all(engine)

print("Tables created successfully!")

In [ ]:
conn = psycopg2.connect(DATABASE_URL)
cursor = conn.cursor()

# Query to list tables
cursor.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public';")
tables = cursor.fetchall()

print("Tables in the database:")
for table in tables:
    print(table[0])

cursor.close()
conn.close()

In [ ]:
table_names = ['manas_user', 'alfreda_user', 'roles', 'university']
conn = psycopg2.connect(DATABASE_URL)
cursor = conn.cursor()

for table_name in table_names:
    try:
        print(f"\nEntries in table: {table_name}")
        cursor.execute(f"SELECT * FROM {table_name};")
        records = cursor.fetchall()

        # Print column names
        colnames = [desc[0] for desc in cursor.description]
        print(", ".join(colnames))

        for row in records:
            print(row)

    except psycopg2.Error as e:
        print(f"Error fetching data from {table_name}: {e}")

cursor.close()
conn.close()